# Main evaluation — all models → review sheet → final results (plan B)

Run after the Pilot has been frozen (`benchmark/evaluation_mode.json` + tag `v1.0-pilot-frozen`).

| Step | What | Output (in `OUT_DIR`) |
|---|---|---|
| 1 | Settings: models, Main definition, frozen mode | — |
| 2 | Check the Main definition (closed vocabulary, links) | lint report |
| 3 | Run all evaluators for every model | `raw/<Model>_main_results.json/.csv` |
| 4 | Apply auto / human, build the **review sheet** | `review_sheet.csv`, `main_state.json` |
| 5 | People fill `review_sheet.csv` (P / F / U), upload it back | — |
| 6 | Merge, audit report, final tables, download everything | `final/…`, `tables/…`, `main_outputs.zip` |

Steps 1–4 and 5–6 can run in different Colab sessions: STEP 4 saves `main_state.json`, STEP 5 reloads it.

## STEP 0 — Get the code from GitHub

Clones the repository (first run) or pulls the latest version, then imports `evaluation/common.py` and **every `evaluation/eval_*.py` automatically** — a new evaluator file is picked up without editing this notebook.

- `BRANCH = "v1.0-pilot-frozen"` (the frozen tag) for the real Main run; `"main"` only for a trial; set it to your branch name to test your work before it is merged.
- Private repository only: add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon, left sidebar).
- CPU runtime is enough for evaluation (no GPU needed).

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Soniaaaa-aa/t2m-capability-benchmark"
BRANCH = "main"                                   # real run: "v1.0-pilot-frozen"
REPO_DIR = Path("/content/t2m-capability-benchmark")

def _git(*args, cwd=None):
    print("$ git", " ".join(args))
    subprocess.run(["git", *args], cwd=cwd, check=True)

url = REPO_URL
try:  # optional token for a private repository
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if not (REPO_DIR / ".git").exists():
    _git("clone", "--branch", BRANCH, url, str(REPO_DIR))
else:
    _git("fetch", "origin", cwd=REPO_DIR)
    _git("checkout", BRANCH, cwd=REPO_DIR)
    _git("pull", "origin", BRANCH, cwd=REPO_DIR)

EVAL_DIR = REPO_DIR / "evaluation"
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))

import common
importlib.reload(common)                 # pick up changes after a pull
from common import *                     # settings, loaders, registry, gold helpers, runner
evaluator_modules = load_all_evaluators(EVAL_DIR)   # imports every eval_*.py

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("Framework commit     :", commit or "unknown")

import json, shutil
import finalize as F

def download(path):
    try:
        from google.colab import files
        files.download(str(path))
    except Exception as e:  # not in Colab
        print("(download skipped:", type(e).__name__, ")", path)

## STEP 1 — Settings

* `MODELS`: folder names inside the standardised ZIPs (`<Model>/<prompt_id>.npy`).
  Each model's ZIP is asked for in STEP 3 unless `/content/benchmark_inputs/<Model>/` already exists.
* `BENCHMARK_FILE`: the Main definition in `benchmark/`.

In [ ]:
MODELS = ["MoMADiff", "MotionHiFlow"]                 # ← all models
BENCHMARK_FILE = "main_benchmark_definition.json"     # ← in benchmark/
SPLIT = "main"
OUT_DIR = Path("/content/main_outputs")
MODE_PATH = REPO_DIR / "benchmark" / F.MODE_FILENAME

mode = F.load_mode(MODE_PATH)
if mode.get("missing_file"):
    print("⚠️ benchmark/evaluation_mode.json not found — EVERY requirement goes to human review.")
else:
    print("Evaluation mode from", MODE_PATH.name, "created", mode.get("created_utc"),
          "commit", mode.get("framework_commit"))
    print("auto :", sorted(t for t, m in mode["types"].items() if m["mode"] == "auto"))
    print("human:", sorted(t for t, m in mode["types"].items() if m["mode"] == "human"))
    problems = F.check_frozen(mode)
    for p in problems:
        print("⚠️ threshold mismatch —", p)
    if not problems:
        print("Thresholds in the code match the frozen decision ✅")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## STEP 2 — Check the Main definition

Errors mean the rules cannot judge a requirement (e.g. an action outside the vocabulary
walk / turn / jump / kick / reach / raise_hand / raise_hands). Fix the definition, or accept that those
requirements go to human review (`ALLOW_LINT_ERRORS = True`).

In [ ]:
bench_path = REPO_DIR / "benchmark" / BENCHMARK_FILE
if bench_path.exists():
    benchmark_definition, main_prompts = load_benchmark_definition(bench_path)
else:
    from google.colab import files
    print(f"{BENCHMARK_FILE} is not in benchmark/ — upload it")
    up = files.upload()
    benchmark_definition, main_prompts = parse_benchmark_definition(next(iter(up.values())), next(iter(up)))
    bench_path.parent.mkdir(parents=True, exist_ok=True)
    bench_path.write_bytes(next(iter(up.values())))
print(len(main_prompts), "prompts")
ALLOW_LINT_ERRORS = False
ok = F.print_lint(F.lint_definition(main_prompts))
if not ok and not ALLOW_LINT_ERRORS:
    raise SystemExit("Fix the definition or set ALLOW_LINT_ERRORS = True")

## STEP 3 — Run all evaluators for every model

In [ ]:
all_rows = []
for model in MODELS:
    data = load_inputs(model, REPO_DIR, benchmark_file=BENCHMARK_FILE, verbose=False)
    cases = data["evaluation_cases"]
    missing = len(main_prompts) - len(cases)
    rows = run_all_evaluators(cases)                     # no Human Gold in Main
    save_results(rows, OUT_DIR / "raw" / f"{model}_{SPLIT}_results.json",
                 {"model": model, "split": SPLIT, "framework_commit": commit, "benchmark_file": BENCHMARK_FILE})
    status = {s: sum(r["status"] == s for r in rows) for s in sorted({r["status"] for r in rows})}
    print(f"{model:14s} prompts={len(cases)} (missing {missing})  requirements={len(rows)}  {status}")
    all_rows += rows

## STEP 4 — auto / human and the review sheet

* **human_required**: every requirement of a *human* type, plus any the rules could not decide.
* **audit**: a blind random sample of *auto* requirements — per type 10 % (at least 20), spread over the models.

The sheet does not show the automatic result. Download `review_sheet.csv` and share it (e.g. Google Sheets).

In [ ]:
AUDIT_FRACTION, MIN_AUDIT_PER_TYPE, SEED = 0.10, 20, 0
final_rows = F.apply_mode(all_rows, mode)
items = F.make_review_sheet(final_rows, main_prompts, AUDIT_FRACTION, MIN_AUDIT_PER_TYPE, SEED,
                            gif_pattern="{model}_gifs/{prompt_id}.gif")
sheet_path = F.save_review_sheet(items, OUT_DIR / "review_sheet.csv")
state_path = OUT_DIR / "main_state.json"
F.save_final_rows(final_rows, state_path, {"split": SPLIT, "framework_commit": commit, "models": MODELS})

from collections import Counter
print("Review items:", len(items), dict(Counter(i["why"] for i in items)))
for t, n in sorted(Counter(i["type"] for i in items).items()):
    print(f"   {t:16s} {n}")
download(sheet_path)
download(state_path)

## STEP 5 — Upload the filled review sheet

Instructions for reviewers (also in the team runbook):

1. Open the GIF named in column `gif` (from each model's `<Model>_gifs_<split>.zip`).
2. Judge **only** the requirement in `type` / `value` for that prompt — not the whole prompt.
3. `human_label`: **P** = satisfied, **F** = not satisfied, **U** = cannot tell. Optional `note`.
4. Do not change any other column. Export as CSV.

If this is a new Colab session, run STEP 0 and STEP 1 first; `main_state.json` is loaded from `OUT_DIR`
or asked for.

In [ ]:
state_path = OUT_DIR / "main_state.json"
if not state_path.exists():
    from google.colab import files
    print("Upload main_state.json (from STEP 4)")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    state_path.write_bytes(next(iter(files.upload().values())))
final_rows = json.loads(state_path.read_text(encoding="utf-8"))["results"]
_, main_prompts = load_benchmark_definition(REPO_DIR / "benchmark" / BENCHMARK_FILE)

filled = OUT_DIR / "review_sheet_filled.csv"
if not filled.exists():
    from google.colab import files
    print("Upload the FILLED review sheet (CSV)")
    filled.write_bytes(next(iter(files.upload().values())))
reviews, problems = F.load_review_sheet(filled)
for p in problems:
    print("⚠️", p)
status = F.merge_reviews(final_rows, reviews)
print(len(reviews), "labels read;", status["pending_human"], "human_required requirements still without P/F")

## STEP 6 — Audit, final tables, download

* **audit report**: agreement of the rules with people on the audit sample (plan §6.6, report in the paper).
* **tables**: per model — overall, by capability, by difficulty, by requirement type.
  Requirement pass rate over decided labels; prompt success = all requirements of the prompt PASS.

In [ ]:
audit = F.audit_report(final_rows)
tables = F.score_tables(final_rows, main_prompts)
extra = {"audit": [dict(requirement_type=t, **s) for t, s in audit.items()],
         "evaluation_mode": [dict(requirement_type=t, mode=m["mode"], reason=m["reason"])
                             for t, m in mode.get("types", {}).items()]}
print("\nFiles:", F.save_tables(tables, OUT_DIR / "tables", extra))
F.save_final_rows(final_rows, OUT_DIR / "final" / f"final_{SPLIT}_results.json",
                  {"split": SPLIT, "framework_commit": commit, "models": MODELS,
                   "mode_created": mode.get("created_utc")})
print((OUT_DIR / "tables" / "summary.md").read_text(encoding="utf-8")[:3000])

archive = shutil.make_archive(str(OUT_DIR.parent / "main_outputs"), "zip", OUT_DIR)
print("Archive:", archive)
download(archive)